# O-information

Min/max O-information of hidden-unit subsets, het vs hom, from the per-subset records. Emits F4_5_minmax_oinfo_clean.png and tab_oinfo.tex.

In [ ]:
import sys
sys.path.append("../code")
import json
import numpy as np
import matplotlib.pyplot as plt
import common as C, stats as st, plots as P
P.apply()

## subset sweep

In [ ]:
# subset sweep; per-subset records for the first ten seeds ship in
# cache/results/os_info_records.jsonl, this regenerates the n20 records from checkpoints. high-k tail dropped: min/max uses k <= 10.
SUMMARY = C.RESULTS / "oinfo_summary.json"
RECORDS = C.RESULTS / "os_info_records_n20.jsonl"

if SUMMARY.exists():
    print("oinfo summary present, skipping sweep")
else:
    import hashlib, math, time
    import torch
    from syntropy.discrete.utils import construct_joint_distribution
    from syntropy.discrete import o_information, s_information
    sys.path.append("../model")
    from model import RSNN
    from data_gen import open_file

    DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cuda.matmul.allow_tf32 = False
    STRIDE = 4
    N_TRIALS = 2048                     # 8 batches x 256, the 512000-column cache
    SUBSET_SIZES = [2, 4, 6, 8, 10, 12]
    N_SUBSETS = 50
    MIN_SUBSETS = 5

    def _ss(*p):
        return int(hashlib.sha256("|".join(map(str, p)).encode()).hexdigest()[:8], 16)

    def os_info(sub):
        jd = construct_joint_distribution(np.asarray(sub, dtype=int))
        O = float(o_information(jd)[1])
        S = float(s_information(jd)[1])
        return O, S, (S + O) / 2.0, (S - O) / 2.0, len(jd)

    def batches(units, times, prms):
        bs, nb_steps, nb_in = int(prms["batch_size"]), int(prms["nb_steps"]), int(prms["nb_inputs"])
        inv_dt = 1.0 / prms["time_step"]
        for c in range(len(times) // bs):
            x = torch.zeros(bs, nb_steps, nb_in)
            for j in range(bs):
                i = bs * c + j
                ts = np.round(np.asarray(times[i]) * inv_dt).astype(np.int64)
                ok = ts < nb_steps
                x[j, ts[ok], np.asarray(units[i], dtype=np.int64)[ok]] = 1.0
            yield x

    @torch.no_grad()
    def build_cache(seed, arch):
        out = C.spk_path(seed, arch)
        if out.exists():
            return
        ck = torch.load(C.ckpt_path(seed, arch), map_location=DEV, weights_only=False)
        prms = dict(ck["prms"])
        prms.update({"dtype": torch.float, "device": DEV, "cuda": DEV.type == "cuda"})
        model = RSNN(prms, rec=True).to(DEV)
        model.load_state_dict(ck["model_state_dict"])
        model.eval()
        nb_hidden = int(model.network[0].output_size)
        chunks, n = [], 0
        for x in batches(UNITS, TIMES, prms):
            spk = model(0, 0, x.to(DEV))[0][2]              # [B, T, 64] binary hidden spikes
            spk = spk[:, ::STRIDE, :].detach().cpu().numpy()
            chunks.append(np.transpose(spk, (2, 0, 1)).reshape(nb_hidden, -1))
            n += spk.shape[0]
            if n >= N_TRIALS:
                break
        spk = np.concatenate(chunks, axis=1)[:, :N_TRIALS * (int(prms["nb_steps"]) // STRIDE)]
        assert np.all((spk == 0) | (spk == 1))
        np.save(out, spk.astype(np.uint8))
        del model
        if DEV.type == "cuda":
            torch.cuda.empty_cache()

    def sweep_net(seed, arch, done, rec_file):
        spk = np.load(C.spk_path(seed, arch)).astype(int)
        active = np.where(spk.sum(1) > 0)[0]
        spk = spk[active]
        n_active = spk.shape[0]
        source = "hom_nowd" if arch == "hom" and seed in C.NOWD_SEEDS else arch
        jobs = []
        for k in SUBSET_SIZES:
            if k > n_active:
                continue
            cap = max(min(N_SUBSETS, math.comb(n_active, k)), min(MIN_SUBSETS, math.comb(n_active, k)))
            rng = np.random.default_rng(_ss(seed, arch, k))
            seen, subs = set(), []
            while len(subs) < cap:
                s = tuple(sorted(rng.choice(n_active, size=k, replace=False).tolist()))
                if s not in seen:
                    seen.add(s)
                    subs.append(s)
            jobs += [(k, i, s) for i, s in enumerate(subs)]
        todo = [(k, i, s) for (k, i, s) in jobs if (seed, arch, k, i) not in done]
        t0 = time.time()
        for k, i, sub in todo:
            try:
                O, S, TC, DTC, nstates = os_info(spk[list(sub)])
                rec = {"seed": seed, "arch": arch, "k": k, "subset_draw": i, "O_info": O, "S_info": S,
                       "TC": TC, "DTC": DTC, "n_states": nstates, "n_active": int(n_active),
                       "is_max": (k == n_active), "status": "ok", "neurons": [int(x) for x in sub],
                       "source": source}
            except Exception as exc:
                rec = {"seed": seed, "arch": arch, "k": k, "subset_draw": i, "n_active": int(n_active),
                       "status": f"error:{type(exc).__name__}", "source": source}
            rec_file.write(json.dumps(rec) + "\n")
            rec_file.flush()
        print(f"seed {seed} {arch}: {len(todo)} subsets (n_active={n_active}) in {time.time() - t0:.0f}s")

    print("loading shd test set ...")
    u, t, _l = open_file(str(C.SHD / "shd_test.h5"))
    UNITS, TIMES = list(u[:]), list(t[:])
    u._v_file.close()

    done = set()
    if RECORDS.exists():
        for line in open(RECORDS):
            try:
                r = json.loads(line)
                done.add((r["seed"], r["arch"], r["k"], r["subset_draw"]))
            except Exception:
                pass

    jobs = [(s, a) for s in C.SEEDS_NEW for a in ("het", "hom")] + [(s, "hom") for s in sorted(C.NOWD_SEEDS)]
    rec_file = open(RECORDS, "a")
    try:
        for seed, arch in jobs:
            build_cache(seed, arch)
            sweep_net(seed, arch, done, rec_file)
    finally:
        rec_file.close()

## min/max O-information

In [ ]:
RECORDS_N10 = C.RESULTS / "os_info_records.jsonl"
RECORDS_N20 = C.RESULTS / "os_info_records_n20.jsonl"
RK = [2, 4, 6, 8, 10]                  # k > 10 is bias-dominated and unevenly swept


def load_jsonl(path):
    return [json.loads(l) for l in open(path) if l.strip()] if path.exists() else []


def load_recs():
    recs = [r for r in load_jsonl(RECORDS_N10) if r.get("status") == "ok"]
    recs += [r for r in load_jsonl(RECORDS_N20) if r.get("status") == "ok"]
    return recs


def minmax(recs):
    # pairing resolved per k: sweep coverage is not uniform across k
    seeds = {"het": set(), "hom": set()}
    store = {}
    for r in recs:
        store.setdefault((r["arch"], r["seed"], r["k"]), []).append(r["O_info"])
        seeds[r["arch"]].add(r["seed"])
    paired = sorted(seeds["het"] & seeds["hom"])
    agg = {a: {"min": {}, "max": {}, "spread": {}} for a in ("het", "hom")}
    seeds_by_k = {}
    for k in RK:
        pk = [s for s in paired if store.get(("het", s, k)) and store.get(("hom", s, k))]
        seeds_by_k[k] = pk
        for a in ("het", "hom"):
            mn = np.array([min(store[(a, s, k)]) for s in pk])
            mx = np.array([max(store[(a, s, k)]) for s in pk])
            agg[a]["min"][k] = mn
            agg[a]["max"][k] = mx
            agg[a]["spread"][k] = mx - mn
    return agg, paired, seeds_by_k


def oinfo_figs():
    recs = load_recs()
    agg, paired, seeds_by_k = minmax(recs)
    n_by_k = {k: len(seeds_by_k[k]) for k in RK}
    n = len(paired)
    print(f"paired seeds: n={n}")

    fig, ax = plt.subplots(figsize=(8.4, 5.4))
    for a in ("het", "hom"):
        mx = np.array([agg[a]["max"][k].mean() for k in RK])
        mn = np.array([agg[a]["min"][k].mean() for k in RK])
        ax.plot(RK, mx, color=P.COL[a], lw=2, marker="^", label=f"{P.LABEL[a]}: max O")
        ax.plot(RK, mn, color=P.COL[a], lw=2, ls="--", marker="v",
                label=f"{P.LABEL[a]}: min O")
        ax.fill_between(RK, mn, mx, color=P.COL[a], alpha=0.10, lw=0)
    ax.axhline(0, color=P.CHANCE, lw=1.2, ls=":")
    ax.set_xlabel("subset size k")
    ax.set_ylabel("O-information")
    uniform = len(set(n_by_k[k] for k in RK)) == 1
    ax.set_title(f"Min/max O-information of subsets, n = {n}")
    if not uniform:
        for k in RK:
            ax.annotate(f"n={n_by_k[k]}", xy=(k, ax.get_ylim()[0]), xytext=(0, 4),
                        textcoords="offset points", ha="center", fontsize=7, color=P.CHANCE)
    ax.set_xticks(RK)
    ax.legend(fontsize=8)
    P.save(fig, C.FIGDIR / "F4_5_minmax_oinfo.png")

    tables = {8: [], 10: []}
    for k in RK:
        for stat in ("min", "max", "spread"):
            h, o = agg["het"][stat][k], agg["hom"][stat][k]
            if h.size != o.size or h.size == 0:
                continue
            r = st.paired_triple(h, o, f"O {stat} k={k}")
            if k in tables:
                tables[k].append([stat, f"{r['het_mean']:+.3f}", f"{r['hom_mean']:+.3f}",
                                  f"{r['delta']:+.3f}", f"{r['dz']:+.2f}",
                                  P.fmt_p(r["perm_p"], r["perm_floor"]), f"{r['wins']}/{r['n']}"])
            print(f"k={k:>2} {stat:>7} | het {r['het_mean']:+.3f} hom {r['hom_mean']:+.3f} "
                  f"delta {r['delta']:+.3f} p {P.fmt_p(r['perm_p'], r['perm_floor'])} wins {r['wins']}/{r['n']}")

    def frac_pos(a, k):
        pk = set(seeds_by_k[k])
        v = [r["O_info"] for r in recs if r["arch"] == a and r["k"] == k and r["seed"] in pk]
        return float(np.mean(np.array(v) > 0)) if v else float("nan")

    fp = {k: {a: frac_pos(a, k) for a in ("het", "hom")} for k in (8, 10) if k in RK}
    summary = {"n_paired": n, "paired_seeds": paired,
               "n_by_k": {str(k): n_by_k[k] for k in RK},
               "frac_pos_k8": fp.get(8, {}), "frac_pos_k10": fp.get(10, {}),
               "k8_table": tables[8], "k10_table": tables[10],
               "k8": {stat: {"het": float(agg["het"][stat][8].mean()),
                             "hom": float(agg["hom"][stat][8].mean())} for stat in ("min", "max", "spread")},
               "k10": {stat: {"het": float(agg["het"][stat][10].mean()),
                              "hom": float(agg["hom"][stat][10].mean())} for stat in ("min", "max", "spread")}}
    out = C.RESULTS / "oinfo_summary.json"
    if not out.exists():
        json.dump(summary, open(out, "w"), indent=1)
    return summary


oinfo_summary = oinfo_figs()

## tab_oinfo.tex

In [ ]:
OI = json.load(open(C.RESULTS / "oinfo_summary.json"))
_k8 = {r[0]: r for r in OI["k8_table"]}


def _orow(label, stat):
    r = _k8[stat]
    return f"    {label} & {r[1]} & {r[2]} & {r[3]} & {r[4]} & {r[5]} & {r[6]} \\\\"


tex = (
    "\\begin{table}[H]\n  \\centering\n  \\small\n  \\setlength{\\tabcolsep}{4pt}\n"
    "  \\caption{O-information of eight-neuron subsets, twenty paired seeds: the most redundant subset, "
    "the most synergistic subset and their range, with the paired difference and the sign-flip test.}\n"
    "  \\label{tab:oinfo}\n"
    "  \\resizebox{\\linewidth}{!}{%\n"
    "  \\begin{tabular}{lrrrrrr}\n    \\toprule\n"
    "    Quantity & het & hom & $\\Delta$ & $d_z$ & sign-flip $p$ & wins \\\\\n    \\midrule\n"
    + _orow(r"Most redundant subset (max $O$)", "max") + "\n"
    + _orow(r"Most synergistic subset (min $O$)", "min") + "\n"
    + _orow(r"Range (max $-$ min)", "spread") + "\n"
    "    \\bottomrule\n  \\end{tabular}}\n\\end{table}\n"
)
HDR = "% auto-generated by notebooks/05_oinfo.ipynb; regenerate rather than edit.\n\n"
(C.TABDIR / "tab_oinfo.tex").write_text(HDR + tex, encoding="utf-8")
print(tex)